# 01 - Data Overview
## Dataset de maladies cardiovasculaires

Ce notebook a pour objectif principal de présenter le dataset cardiovasculaire et d’établir une vue d’ensemble de ses caractéristiques. Il sert de première étape avant toute analyse plus poussée, en permettant de comprendre la structure des données, la nature des variables et la qualité globale du jeu de données.

### Présentation du dataset

Le jeu de données contient des informations cliniques et démographiques sur des patients évalués dans un contexte de risque cardiovasculaire. Il regroupe plusieurs types de variables :
- données démographiques : âge, sexe ;
- mesures cliniques : pression artérielle au repos, cholestérol, fréquence cardiaque maximale ;
- symptômes et indicateurs médicaux : type de douleur thoracique, résultats de l’électrocardiogramme, angine induite par l’effort ;
- variable cible : présence ou absence de maladie cardiaque.

### Objectifs de ce notebook

Dans ce notebook, nous allons :
1. examiner la structure du dataset ;
2. vérifier les types de colonnes et la qualité des données ;
3. détecter les anomalies et les valeurs incohérentes ;
4. observer la distribution de la variable cible ;
5. poser les bases pour les prochaines étapes de prétraitement et de modélisation.

Cette première vue d’ensemble est essentielle pour garantir une compréhension solide du dataset avant d’aller plus loin dans l’analyse.

---

In [13]:
# Cellule 1: Initialisation et imports
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Imports réussis!")

✅ Imports réussis!


## 1. Téléchargement et chargement du dataset

Dans cette étape, nous vérifions si le fichier de données est déjà présent localement. Sinon, il est téléchargé puis chargé dans un DataFrame pandas pour démarrer l’analyse et préparer la vue d’ensemble.

In [14]:
# Download the dataset from KaggleHub if it doesn't exist
import os
import shutil
import kagglehub
import pandas as pd

# 1. Définition des chemins
notebook_dir = os.getcwd()
data_dir = os.path.join(notebook_dir, "data")
os.makedirs(data_dir, exist_ok=True)
destination_file = os.path.join(data_dir, "cardiovascular.csv")

# 2. Récupérer le fichier uniquement s'il n'existe pas encore
if not os.path.exists(destination_file):
    print("Fichier absent, téléchargement via KaggleHub...")
    downloaded_path = kagglehub.dataset_download("jocelyndumlao/cardiovascular-disease-dataset")
    
    # Trouver le fichier source
    source_file = None
    for root, dirs, files in os.walk(downloaded_path):
        for f in files:
            if f.endswith(".csv") or "." not in f:
                source_file = os.path.join(root, f)
                break
        if source_file:
            break
            
    if source_file:
        shutil.copy2(source_file, destination_file)
        print(f"Fichier copié vers : {destination_file}")
    else:
        raise FileNotFoundError("Aucun fichier valide trouvé dans le téléchargement Kaggle.")
else:
    print(f"Fichier déjà existant : {destination_file}")

# 3. Chargement systématique
df = pd.read_csv(destination_file)
print(df.head())

Fichier déjà existant : c:\Users\Omar\Desktop\Projects\Cardiovascular Disease\notebooks\data\cardiovascular.csv
   patientid  age  gender  chestpain  restingBP  serumcholestrol  \
0     103368   53       1          2        171                0   
1     119250   40       1          0         94              229   
2     119372   49       1          2        133              142   
3     132514   43       1          0        138              295   
4     146211   31       1          1        199                0   

   fastingbloodsugar  restingrelectro  maxheartrate  exerciseangia  oldpeak  \
0                  0                1           147              0      5.3   
1                  0                1           115              0      3.7   
2                  0                0           202              1      5.0   
3                  1                1           153              0      3.2   
4                  0                2           136              0      5.3   

   s

## 2. Aperçu du dataset

Nous commençons par examiner la taille du dataset, sa structure et le type de chaque colonne afin de vérifier que les variables sont bien interprétées avant toute analyse plus poussée.

In [15]:
print(df.shape)

(1000, 14)


Le dataset contient 1000 observations et 14 colonnes. Cette structure est suffisamment riche pour étudier les relations entre les caractéristiques cliniques et la variable cible.

In [16]:
print(df.columns)

Index(['patientid', 'age', 'gender', 'chestpain', 'restingBP',
       'serumcholestrol', 'fastingbloodsugar', 'restingrelectro',
       'maxheartrate', 'exerciseangia', 'oldpeak', 'slope', 'noofmajorvessels',
       'target'],
      dtype='str')


In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   patientid          1000 non-null   int64  
 1   age                1000 non-null   int64  
 2   gender             1000 non-null   int64  
 3   chestpain          1000 non-null   int64  
 4   restingBP          1000 non-null   int64  
 5   serumcholestrol    1000 non-null   int64  
 6   fastingbloodsugar  1000 non-null   int64  
 7   restingrelectro    1000 non-null   int64  
 8   maxheartrate       1000 non-null   int64  
 9   exerciseangia      1000 non-null   int64  
 10  oldpeak            1000 non-null   float64
 11  slope              1000 non-null   int64  
 12  noofmajorvessels   1000 non-null   int64  
 13  target             1000 non-null   int64  
dtypes: float64(1), int64(13)
memory usage: 109.5 KB


### Observation initiale

À première vue :
- aucune valeur manquante n’apparaît dans les colonnes, mais cette observation doit être confirmée par une vérification plus rigoureuse ;
- les colonnes **chestpain** et **restingrelectro** sont codées numériquement mais correspondent à des variables catégorielles, ce qui devra être pris en compte lors du prétraitement.

In [18]:
df.head()

,patientid,age,gender,chestpain,restingBP,serumcholestrol,fastingbloodsugar,restingrelectro,maxheartrate,exerciseangia,oldpeak,slope,noofmajorvessels,target
0,103368,53,1,2,171,0,0,1,147,0,5.3,3,3,1
1,119250,40,1,0,94,229,0,1,115,0,3.7,1,1,0
2,119372,49,1,2,133,142,0,0,202,1,5.0,1,0,0
3,132514,43,1,0,138,295,1,1,153,0,3.2,2,2,1
4,146211,31,1,1,199,0,0,2,136,0,5.3,3,2,1


### 3. Résumé statistique avec `df.describe()`

Cette méthode permet de visualiser rapidement les statistiques descriptives du dataset, notamment la moyenne, l’écart-type, les quartiles et les valeurs extrêmes. Elle est utile pour détecter des anomalies de plage ou des valeurs incohérentes.

In [19]:
print(df.describe())

          patientid         age       gender    chestpain    restingBP  \
count  1.000000e+03  1000.00000  1000.000000  1000.000000  1000.000000   
mean   5.048704e+06    49.24200     0.765000     0.980000   151.747000   
std    2.895905e+06    17.86473     0.424211     0.953157    29.965228   
min    1.033680e+05    20.00000     0.000000     0.000000    94.000000   
25%    2.536440e+06    34.00000     1.000000     0.000000   129.000000   
50%    4.952508e+06    49.00000     1.000000     1.000000   147.000000   
75%    7.681877e+06    64.25000     1.000000     2.000000   181.000000   
max    9.990855e+06    80.00000     1.000000     3.000000   200.000000   

       serumcholestrol  fastingbloodsugar  restingrelectro  maxheartrate  \
count      1000.000000        1000.000000      1000.000000   1000.000000   
mean        311.447000           0.296000         0.748000    145.477000   
std         132.443801           0.456719         0.770123     34.190268   
min           0.000000       

### Points de vigilance

- La colonne **serumcholestrol** présente une valeur minimale égale à 0, ce qui est biologiquement incohérent, car les valeurs réalistes observées sont nettement supérieures. Une stratégie de correction sera définie dans les étapes suivantes.
- La colonne **slope** contient également une valeur 0 alors que ses valeurs attendues sont généralement 1, 2 ou 3. Cette anomalie devra être traitée avec soin pour éviter de biaiser l’analyse.

### 4. Vérification des valeurs aberrantes

Nous examinons ici les valeurs hors plage dans les colonnes **serumcholestrol** et **slope** afin de confirmer l’existence de données manquantes ou mal saisies et de déterminer la meilleure méthode de traitement avant la modélisation.

In [20]:
df[['serumcholestrol']].value_counts().sort_index()

serumcholestrol
0                  53
85                  1
86                  4
87                  2
132                 6
                   ..
559                 3
560                 1
561                 2
601                 3
602                 2
Name: count, Length: 344, dtype: int64

La colonne **serumcholestrol** contient des valeurs comme 0, 85, 86 et 87, qui ne sont pas cohérentes avec les niveaux habituels observés dans ce type de dataset.
> Ces valeurs sont atypiques et devront être traitées selon une stratégie adaptée avant la modélisation.

In [21]:
df[["slope"]].value_counts()

slope
2        322
1        299
3        199
0        180
Name: count, dtype: int64

La méthode `value_counts()` montre que la colonne **slope** comporte plusieurs valeurs, notamment 0, 1, 2, 3 et 4. Or, la valeur 0 est hors plage par rapport au codage attendu.
> Cette observation suggère que la valeur 0 correspond très probablement à une donnée incorrecte ou à une erreur de saisie.

In [22]:
df[["target"]].value_counts()

target
1         580
0         420
Name: count, dtype: int64

La variable cible est relativement bien équilibrée entre les classes, ce qui est favorable pour l'entraînement d'un modèle de classification sans biais excessif.

### 5. Vérification des doublons

Nous vérifions maintenant qu'il n'existe pas de lignes redondantes dans le dataset, car des doublons peuvent fausser les statistiques et artificiellement améliorer ou dégrader la performance des modèles.

In [24]:
df.duplicated().sum()

np.int64(0)

À ce stade, aucune ligne redondante n'est détectée dans le dataset. Le jeu de données semble propre sur le plan des doublons, ce qui facilite la suite de l'analyse.

### Conclusion de cette étape

Ce notebook fournit une vue d’ensemble du dataset et vérifie la qualité de base des données. Il pose la base nécessaire pour les prochaines étapes de nettoyage, de préparation et de modélisation.

---